# Fault Tolerance and Observability

## TLDR

Long training runs fail. Hardware faults, preemptions, and out-of-memory kills
are normal at scale, so a real system has to survive them and let you see what is
happening. In this notebook you make a training loop checkpoint aware, then watch
Ray Train recover from a worker you kill on purpose mid-run. After that you
measure throughput and profile the loop, and tour the Anyscale tools that make a
distributed run observable.


## Introduction

Everything since notebook 01 has called `ray.train.report` with a checkpoint.
Now that checkpoint earns its keep. Ray Train gives you two recovery mechanisms.
Automatic retries detect a dead worker, rebuild the worker group, and resume from
the latest checkpoint, all driven by a `FailureConfig`. Manual restoration lets
you re-create the trainer later and pick up where it left off. Both depend on one
thing, a training loop that loads a checkpoint on startup if one exists.

We prove it the honest way. We schedule a real `SIGKILL` of a training worker a
few seconds into the run, the same kind of abrupt death a preemption causes, and
watch the job come back and finish. Then we turn to observability. You cannot fix
what you cannot see, so we measure throughput, profile the loop with the PyTorch
profiler, and look at the Anyscale tools built for distributed training.

<div align="center"><img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/fault_tolerance_train_v2.png" width="90%" loading="lazy"></div><br>

## Key concepts used in this notebook

**Checkpoint-aware loop.** Call `ray.train.get_checkpoint` at the top. If it
returns a checkpoint, load the model, optimizer, and epoch and resume. If not,
start fresh.

**`FailureConfig`.** Set `max_failures` to the number of automatic retries. On a
worker death Ray Train rebuilds the group and resumes from the last checkpoint
instead of raising.

**Manual restoration.** Re-create the trainer with the same `name` and
`storage_path` and call `fit` again to continue a run after the cluster came down.

**Elastic training.** Give `num_workers` a range so the job keeps going with
fewer workers when nodes are lost and grows back when they return.

**Throughput.** Rows per second. Global throughput is roughly per-worker
throughput times the number of workers, so it should scale close to linearly.

**Profiling.** The PyTorch profiler records where time and memory go, by
operator, kernel, and over the timeline.


## What you will learn

- How to make a training loop resume from a checkpoint
- How `FailureConfig` turns a worker death into an automatic recovery
- How to watch a live kill and recovery, and read the two attempts
- How to resume a finished or interrupted run by hand
- How to measure training throughput and reason about its scaling
- How to profile the loop and where to look in the Anyscale tools


## Why Ray on Anyscale for resilience and observability

| Challenge | Without Ray | With Ray on Anyscale |
|---|---|---|
| Recover from a dead worker | Custom retry and restore logic | `FailureConfig(max_failures=...)` |
| Resume after a crash | Manual bookkeeping of progress | Same `name` and `storage_path`, call `fit` |
| Survive lost nodes | Rewrite for a new world size | Elastic `num_workers=(min, max)` |
| See per-worker state | Stitch logs together by hand | Anyscale Train dashboard, persisted per run |
| Profile GPUs on demand | Edit code, redeploy | Turn on the dynolog daemon by env var |


## Architecture

```
   attempt 1                         worker dies (SIGKILL)
   GPU0 ----train----> checkpoint ----X
   GPU1 ----train----> checkpoint
                          |
                          |  Ray Train detects the death,
                          |  rebuilds the group, hands back
                          |  the latest checkpoint
                          v
   attempt 2
   GPU0 --load ckpt--> resume ----train----> done
   GPU1 --load ckpt--> resume ----train----> done
```

The checkpoint on `/mnt/cluster_storage` is the hinge. Without it a restart would
begin from scratch. With it, the new workers resume from the last saved epoch.


## How this scales on Anyscale

| | This notebook | Production |
|---|---|---|
| Failure | We kill one worker on purpose | Real preemptions and hardware faults |
| Workers | 2 on one node | Many across nodes, elastic ranges |
| Recovery | Resume from the last epoch | Same, plus mid-epoch resumption |
| Visibility | Printed metrics and a local profile | Persisted dashboard and on-demand GPU traces |


## Cell 1 — Connect and stage the data

**What you do.** Connect to Ray and download MNIST once to shared storage so the
workers do not race to download it.

**What to check.** Four GPUs are available. The data lands in
`/mnt/cluster_storage/data`.

**Why it matters.** Staging shared data once is the clean pattern on a cluster.
Every worker reads the same copy.


In [ ]:
import os
import subprocess
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

# 0% external downloads. Stage MNIST from the tutorial\'s PUBLIC S3 mirror
# (unsigned) into shared storage, in torchvision\'s expected layout
# (/mnt/cluster_storage/data/MNIST/raw/...), so no node hits the torchvision CDN.
MIRROR = "s3://anyscale-public-materials/ray_summit_foundation_model_training_2026"
subprocess.run(["aws", "s3", "sync", f"{MIRROR}/datasets/mnist",
                "/mnt/cluster_storage/data/MNIST",
                "--no-sign-request", "--only-show-errors"], check=True)

from torchvision.datasets import MNIST
MNIST(root="/mnt/cluster_storage/data", train=True, download=False)  # verify staged copy

import ray
from common import utils

if not ray.is_initialized():
    ray.init(address="auto", runtime_env=utils.build_runtime_env())

utils.print_cluster_resources()

## A checkpoint-aware training loop

The only change from a normal loop is at the top. Call `ray.train.get_checkpoint`
and, if there is one, restore the model, optimizer, and the epoch to resume from.
Everything else is the ResNet on MNIST loop you would write anyway. The loop also
times each epoch so we can talk about throughput later.


## Cell 2 — Define the checkpoint-aware loop

**What you do.** Define a loop that resumes from a checkpoint when one exists,
trains, measures rows per second, and saves a checkpoint each epoch.

**What to check.** The restore block at the top reads `model.pt`, `optim.pt`, and
the saved epoch. On a fresh run it is skipped. On a recovery it sets
`start_epoch` past the work already done.

**Why it matters.** This block is the whole basis of fault tolerance. Without it,
a restart repeats finished epochs or starts over.


In [ ]:
import ray.train
import ray.train.torch

def train_loop(config):
    import os, time, tempfile
    import torch
    from torch.nn import CrossEntropyLoss
    from torch.optim import Adam
    from torch.utils.data import DataLoader
    from torchvision.datasets import MNIST
    from torchvision.transforms import Compose, ToTensor, Normalize
    import ray.train, ray.train.torch
    from ray.train import Checkpoint
    from common import utils

    world_rank = ray.train.get_context().get_world_rank()
    world_size = ray.train.get_context().get_world_size()

    model = ray.train.torch.prepare_model(utils.build_resnet18_mnist())
    optimizer = Adam(model.parameters(), lr=1e-3)

    # Resume from a checkpoint if one exists. This is the heart of recovery.
    start_epoch = 0
    checkpoint = ray.train.get_checkpoint()
    if checkpoint:
        with checkpoint.as_directory() as ckpt_dir:
            model.module.load_state_dict(torch.load(os.path.join(ckpt_dir, "model.pt")))
            optimizer.load_state_dict(torch.load(os.path.join(ckpt_dir, "optim.pt")))
            start_epoch = torch.load(os.path.join(ckpt_dir, "extra.pt"))["epoch"] + 1
    if world_rank == 0:
        print(f"Starting at epoch {start_epoch}", flush=True)

    transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
    data = MNIST(root="/mnt/cluster_storage/data", train=True, download=False, transform=transform)
    loader = ray.train.torch.prepare_data_loader(
        DataLoader(data, batch_size=256, shuffle=True, drop_last=True)
    )
    criterion = CrossEntropyLoss()

    for epoch in range(start_epoch, config["epochs"]):
        if world_size > 1:
            loader.sampler.set_epoch(epoch)
        model.train()
        rows, start = 0, time.perf_counter()
        for images, labels in loader:
            loss = criterion(model(images), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            rows += images.size(0)
        torch.cuda.synchronize()
        per_worker = rows / (time.perf_counter() - start)

        with tempfile.TemporaryDirectory() as ckpt_dir:
            ckpt = None
            if world_rank == 0:
                torch.save(model.module.state_dict(), os.path.join(ckpt_dir, "model.pt"))
                torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optim.pt"))
                torch.save({"epoch": epoch}, os.path.join(ckpt_dir, "extra.pt"))
                ckpt = Checkpoint.from_directory(ckpt_dir)
            ray.train.report(
                {"loss": float(loss.item()), "epoch": epoch,
                 "rows_per_sec_worker": round(per_worker, 1),
                 "rows_per_sec_global": round(per_worker * world_size, 1)},
                checkpoint=ckpt,
            )


## Cell 3 — Schedule a kill, then launch with retries

**What you do.** Schedule a SIGKILL of one worker about 25 seconds in, then
launch a 2-worker job with `FailureConfig(max_failures=2)`. The `kill_in` helper
waits until workers exist, then kills one.

**What to check.** You will see `Starting at epoch 0`, then the kill message, then
NCCL connection errors from the surviving worker as it notices its peer vanish,
then a second `Starting at epoch N` where N is past the last checkpoint. The run
finishes at the final epoch.

**Why it matters.** This is a real abrupt death, not a clean shutdown, and Ray
Train recovers from it automatically and resumes from the checkpoint.


In [ ]:
from ray.train import ScalingConfig, RunConfig, FailureConfig
from ray.train.torch import TorchTrainer
from common.kill_train_worker import kill_in

run_name = "resnet_mnist_fault_tolerance"
storage_path = "/mnt/cluster_storage/ray_summit_fault_tolerance"

kill_in(seconds=25)   # a worker dies ~25s in, after at least one checkpoint

trainer = TorchTrainer(
    train_loop_per_worker=train_loop,
    train_loop_config={"epochs": 10},
    scaling_config=ScalingConfig(num_workers=2, use_gpu=True),
    run_config=RunConfig(
        storage_path=storage_path, name=run_name,
        failure_config=FailureConfig(max_failures=2),
    ),
)
result = trainer.fit()
print("Final metrics:", result.metrics)


## What just happened

The run made two attempts under one name. Attempt one trained until the worker
was killed. Ray Train saw the death, and because the failure count was under
`max_failures`, it rebuilt the worker group rather than raising. Attempt two
called `get_checkpoint`, found the last saved epoch, and resumed from there. The
final metrics reflect the total epochs trained, not the wall-clock restart.

In the Anyscale Train dashboard this shows as two attempts under the same run, and
the second attempt inherits the checkpoint from the first.

If retries are exhausted, or the whole cluster came down, you recover by hand.
Re-create the trainer with the same `name` and `storage_path` and call `fit`
again. If the run already finished, it returns the final result.


## Cell 4 — Manual restoration

**What you do.** Re-create the trainer with the same name and storage path and
call `fit`.

**What to check.** Because the run above finished all epochs, this returns the
final result without retraining.

**Why it matters.** Resuming a long job after an interruption is the same one
line, `fit`, against the same run name.


In [ ]:
restored_trainer = TorchTrainer(
    train_loop_per_worker=train_loop,
    train_loop_config={"epochs": 6},
    scaling_config=ScalingConfig(num_workers=2, use_gpu=True),
    run_config=RunConfig(storage_path=storage_path, name=run_name),
)
restored_result = restored_trainer.fit()
print("Restored metrics:", restored_result.metrics)


## Elastic training

Standard retries restart with the same worker count, which fails if nodes were
permanently lost. Elastic training lets the job continue with fewer workers and
grow back when capacity returns. You give `num_workers` a range.

```python
scaling_config = ScalingConfig(
    use_gpu=True,
    num_workers=(2, 8),   # min and max instead of a fixed count
)
```

Ray Train asks for the maximum at startup, falls back toward the minimum if
needed, restarts on the survivors when a node is lost, and scales back up as
capacity returns. This is what makes spot and preemptible instances practical,
and it can cut cloud cost substantially. Keep the effective global batch size
steady across worker counts with gradient accumulation.


## Mid-epoch resumption

The recovery above resumed at an epoch boundary, so the partial epoch in progress
when the worker died was repeated. Mid-epoch resumption checkpoints the dataset
iterator position alongside the model, so a restart picks up at the exact batch
where it stopped and every sample is seen once per epoch.

```python
# save the iterator position with the checkpoint
state = dataloader.state_dict()

# on resume, restore the iterator to that position
shard = ray.train.get_dataset_shard("train", state_dict=state)
```

It needs a unique row identifier and map-based transforms, and the iterator state
is written asynchronously with little overhead.


## Observability, part one, throughput

The metrics above already carry throughput. Each worker reported its rows per
second, and the global figure is that times the worker count. The useful property
is that global throughput should scale close to linearly with workers, so two
workers do roughly twice the rows per second of one. When it does not, you have a
bottleneck to find, often in data loading or in communication.

The next tools help you find it. Profile the loop to see where time goes, and use
the Anyscale dashboard to see every worker at once.


## Cell 5 — Profile the loop

**What you do.** Run a short job wrapped in the PyTorch profiler. It records a few
steps and writes a trace and a memory timeline to shared storage.

**What to check.** The run completes and writes files under
`/mnt/cluster_storage/ray_summit_fault_tolerance/profile`. You open those in
TensorBoard or the Chrome trace viewer.

**Why it matters.** The profiler is how you turn a slow run into a fast one. It
shows the expensive operators, the GPU kernels, and where the GPU sat idle
waiting for data.


In [ ]:
def profile_train_loop(config):
    import os
    import torch
    from torch.nn import CrossEntropyLoss
    from torch.optim import Adam
    from torch.utils.data import DataLoader
    from torchvision.datasets import MNIST
    from torchvision.transforms import Compose, ToTensor, Normalize
    import ray.train, ray.train.torch
    from common import utils

    world_rank = ray.train.get_context().get_world_rank()
    model = ray.train.torch.prepare_model(utils.build_resnet18_mnist())
    optimizer = Adam(model.parameters(), lr=1e-3)
    transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
    data = MNIST(root="/mnt/cluster_storage/data", train=True, download=False, transform=transform)
    loader = ray.train.torch.prepare_data_loader(
        DataLoader(data, batch_size=256, shuffle=True, drop_last=True)
    )
    criterion = CrossEntropyLoss()

    out_dir = config["profile_dir"]
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
        on_trace_ready=torch.profiler.tensorboard_trace_handler(out_dir, worker_name=f"rank{world_rank}"),
        record_shapes=True, with_stack=True, profile_memory=True,
    ) as prof:
        model.train()
        for step, (images, labels) in enumerate(loader):
            loss = criterion(model(images), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            prof.step()
            if step >= 6:
                break

    if world_rank == 0:
        prof.export_memory_timeline(os.path.join(out_dir, "memory_rank0.html"))
        print("wrote profile traces to", out_dir, flush=True)
    ray.train.report({"profiled": True})

profile_dir = "/mnt/cluster_storage/ray_summit_fault_tolerance/profile"
profile_trainer = TorchTrainer(
    train_loop_per_worker=profile_train_loop,
    train_loop_config={"profile_dir": profile_dir},
    scaling_config=ScalingConfig(num_workers=2, use_gpu=True),
    run_config=RunConfig(storage_path="/mnt/cluster_storage/ray_summit_profile",
                         name="profile_run"),
)
profile_trainer.fit()

import os
print("profile files:", sorted(os.listdir(profile_dir)))


## Reading the profiler

The PyTorch profiler gives several views, which you open in TensorBoard or by
loading the trace JSON in the Chrome trace viewer.

- **Operator view.** Time spent in each operator, with self time excluding
  children and total time including them. This finds the expensive layers.
- **Trace view.** A timeline of CPU threads and GPU streams. Gaps on the GPU
  stream usually mean the GPU is waiting on data loading.
- **Kernel view.** Every GPU kernel, whether it used tensor cores, and how well
  it occupied the GPU. Low occupancy points at work that does not fill the device.
- **Memory view.** The memory timeline, broken down into parameters, gradients,
  optimizer states, and activations. This is where you spot the activation peaks
  that drive out-of-memory errors.

The memory timeline we exported to `memory_rank0.html` is the same picture you
used to compare strategies in notebook 02. An example memory timeline looks like
the figure below, with each band a category of memory over the training step.

![Example PyTorch memory timeline](images/gpu_memory_profile.png)


## Observability, part two, the Anyscale tools

On Anyscale you get observability built for distributed training, not stitched
together from logs.

**The Train dashboard.** A per-run view that persists after the cluster shuts
down. It shows the controller and every worker, with each worker's rank, GPU,
logs, and metrics in one place. Resource utilization and checkpoint timing are
charted per epoch, so a slow checkpoint or an idle GPU is obvious. When a run
makes several attempts, like the recovery above, you see each attempt under the
one run.

**On-demand GPU profiling with dynolog.** Anyscale runs the dynolog telemetry
daemon, so you can capture a GPU trace from a running job without editing code or
adding a profiler. You turn it on by setting two env vars in the worker runtime
environment.

```python
run_config = RunConfig(
    name="my_run",
    worker_runtime_env={
        "env_vars": {"KINETO_USE_DAEMON": "1", "KINETO_DAEMON_INIT_DELAY_S": "5"},
    },
)
```

With the daemon enabled you trigger a trace on demand and inspect it in the same
profiler views above, which is how you debug a slow step on a live run at scale.


## Conclusion

You made a loop checkpoint aware, then watched Ray Train recover from a worker you
killed on purpose, resuming from the last checkpoint and finishing the run. You
saw manual restoration, the elastic worker range, and mid-epoch resumption. Then
you measured throughput, profiled the loop, and toured the Anyscale Train
dashboard and on-demand dynolog profiling.

Ray and Anyscale features you used. `ray.train.get_checkpoint`, `FailureConfig`,
manual restoration by run name, elastic `num_workers`, the PyTorch profiler under
Ray Train, and the Anyscale dashboard and dynolog daemon.

You now have the full toolkit. Scale a loop, shard it, split it across two axes,
and keep it alive and observable. In notebook 05 we put it all together, and reason
about combining all five parallelism axes with a planner.
